# 08 — Carteras 2025

**Práctica B3-T4 · Forecasting financiero multivariante (SP500, 23 activos)**

Backtest comparativo de **6 carteras** durante 2025 usando el mejor modelo de predicción a 90 días (`mixto_profMIX_in90_out90`, `mae_test = 0.001277`) y el **ensemble top-3 del deep dive**.

## Diseño experimental

```mermaid
flowchart TD
    A["Modelo: mixto_profMIX_in90_out90 + ensemble top-3"] --> B[Loop sobre fechas de rebalance en 2025]
    B --> C["X = ultimos 90 dias de returns hasta fecha rebalance"]
    C --> D["Prediccion: media de retornos siguientes 90 dias por activo"]
    D --> E1["Sin pred: EW, MinVar, MaxSharpe historico"]
    D --> E2["Con pred: Top-K, Momentum tilt, Black-Litterman"]
    E1 --> F[Hold portafolio hasta siguiente rebalance]
    E2 --> F
    F --> G[Compute realized returns]
    G --> H["Metricas: cum return, vol, Sharpe, MDD, IR, TE, turnover"]
    H --> I[Comparativa final]
```

## Las 6 carteras

### Sin predicciones (baselines honestos)

| Estrategia | Justificación |
|---|---|
| **EW** equiponderada | Pesos fijos `1/23`. Benchmark neutral. |
| **MinVar** | Minimiza `w' Σ w`; covarianza muestral últimos 252 días. Cap por activo 15%. |
| **Max-Sharpe histórico** | Maximiza Sharpe usando `μ` y `Σ` históricas. |

### Con predicciones (usan el modelo)

| Estrategia | Justificación |
|---|---|
| **Top-K=5 simple** | Top-5 activos por retorno predicho, pesos iguales. Recomendado en sección 15.2 de la guía del profesor. |
| **Momentum tilt** | `μ_tilted = μ_hist + λ * pred_return`; maximiza Sharpe con `μ_tilted`. Adapta celda 94 del Workshop 1. |
| **Black-Litterman** | Prior CAPM (`δ=2.5`) + views Q del modelo. Adapta celda 97 del Workshop 1. |

## Métricas

- Cumulative return, Annualized return, Annualized vol, **Sharpe**, **Max drawdown**, Information ratio vs EW, Tracking error vs EW, Turnover medio, MAE realizado vs predicho.

## Reglas anti-leakage

- El modelo está entrenado con datos hasta ~2020 (último 10% temporal = test). 2025 es **completamente OOS** para el modelo.
- Para baselines sin predicción (MinVar, MaxSharpe): `μ` y `Σ` calculadas SOLO con los 252 días previos al rebalance (rolling lookback).
- Para baselines con predicción (BL, momentum tilt): mismas `μ`/`Σ` históricas + views del modelo evaluado solo con los últimos 90 días hasta la fecha de rebalance.

## Comparativas

- Rebalance **mensual** vs **trimestral** (8 × 2 + 2 frecuencias EW = 14 backtests, simplificamos a 12 al excluir EW de trimestral por ser constante).
- 2 modelos de predicción: top-1 (`mixto_profMIX_in90_out90`) y ensemble top-3 deep dive.

## 1. Setup

In [ ]:
import os
import json
import time
import pickle
import random
import warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

import tensorflow as tf
from tensorflow.keras.models import load_model

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["savefig.dpi"] = 150
mpl.rcParams["font.size"] = 9
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.3

print(f"TensorFlow {tf.__version__}")


def _resolve_dir(name):
    for cand in [Path(name), Path("..") / name]:
        if cand.exists():
            return cand.resolve()
    p = Path(name)
    p.mkdir(parents=True, exist_ok=True)
    return p.resolve()


RESULTS_DIR = _resolve_dir("results")
MODELS_DIR = _resolve_dir("models")
DATA_DIR = _resolve_dir("data")

N_ASSETS = 23
INPUT_WINDOW = 90
OUTPUT_WINDOW = 90

print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"MODELS_DIR  : {MODELS_DIR}")
print(f"DATA_DIR    : {DATA_DIR}")

## 2. Carga de datos, modelos y baseline de competición

Necesitamos:
- **Retornos log** (input al modelo, idéntico al pipeline de competición).
- **Retornos simples** (para calcular retornos realizados de la cartera; convertimos al final).
- **Precios** (para construir series de NAV de cada cartera).
- **Modelo top-1** `mixto_profMIX_in90_out90.keras`.
- **3 modelos del ensemble**: `mixto_dd_{convLSTM_causal, biGRU_attention, profMIX}_in90_out90.keras`.

In [ ]:
CACHE_RETURNS = DATA_DIR / "sp500_returns.parquet"
CACHE_PRICES = DATA_DIR / "sp500_close.parquet"

TICKERS = [
    "AEP", "BA", "CAT", "CNP", "CVX", "DIS", "DTE", "ED", "GD", "GE",
    "HON", "HPQ", "IBM", "IP", "JNJ", "KO", "KR", "MMM", "MO", "MRK",
    "MSI", "PG", "XOM",
]


def _read(p):
    try:
        return pd.read_parquet(p)
    except Exception:
        with open(p.with_suffix(".pkl"), "rb") as f:
            return pickle.load(f)


if not (CACHE_RETURNS.exists() or CACHE_RETURNS.with_suffix(".pkl").exists()):
    raise FileNotFoundError(
        f"No existe el cache de retornos en {CACHE_RETURNS}. "
        "Ejecuta primero 06_mixtos.ipynb o 07_investigacion.ipynb."
    )

returns_df = _read(CACHE_RETURNS if CACHE_RETURNS.exists() else CACHE_RETURNS.with_suffix(".pkl"))
prices_df = _read(CACHE_PRICES if CACHE_PRICES.exists() else CACHE_PRICES.with_suffix(".pkl"))

common_idx = returns_df.index.intersection(prices_df.index)
returns_df = returns_df.loc[common_idx]
prices_df = prices_df.loc[common_idx]
simple_returns_df = prices_df.pct_change().dropna()

print(f"returns_df (log) : {returns_df.shape}  {returns_df.index.min().date()} -> {returns_df.index.max().date()}")
print(f"prices_df        : {prices_df.shape}")
print(f"simple_returns   : {simple_returns_df.shape}")
print(f"Activos          : {list(returns_df.columns)}")

In [ ]:
MODEL_TOP1_PATH = MODELS_DIR / "mixto_profMIX_in90_out90.keras"
ENSEMBLE_PATHS = {
    "profMIX": MODELS_DIR / "mixto_dd_profMIX_in90_out90.keras",
    "convLSTM_causal": MODELS_DIR / "mixto_dd_convLSTM_causal_in90_out90.keras",
    "biGRU_attention": MODELS_DIR / "mixto_dd_biGRU_attention_in90_out90.keras",
}


def _safe_load(path):
    if not path.exists():
        print(f"  AVISO: no existe {path}")
        return None
    try:
        return load_model(path, compile=False)
    except Exception as exc:
        print(f"  ERROR cargando {path}: {type(exc).__name__}: {exc}")
        return None


model_top1 = _safe_load(MODEL_TOP1_PATH)
print(f"Modelo top-1 cargado: {MODEL_TOP1_PATH.name} -> {'OK' if model_top1 else 'FAIL'}")
if model_top1 is not None:
    print(f"  Input shape: {model_top1.input_shape}    Output shape: {model_top1.output_shape}    Params: {model_top1.count_params():,}")

ensemble_models = {}
for name, path in ENSEMBLE_PATHS.items():
    m = _safe_load(path)
    if m is not None:
        ensemble_models[name] = m
        print(f"Ensemble {name} cargado: {path.name} OK ({m.count_params():,} params)")

print(f"\nResumen: top1={'si' if model_top1 else 'no'}, ensemble_models={len(ensemble_models)}/3")

In [ ]:
YEAR_BACKTEST = 2025

dates_2025 = returns_df.index[returns_df.index.year == YEAR_BACKTEST]
print(f"Dias de trading en {YEAR_BACKTEST}: {len(dates_2025)}")
print(f"  Primer dia: {dates_2025.min().date() if len(dates_2025) > 0 else 'N/A'}")
print(f"  Ultimo dia: {dates_2025.max().date() if len(dates_2025) > 0 else 'N/A'}")

cutoff_start_2024 = pd.Timestamp(f"{YEAR_BACKTEST - 1}-01-01")
hist_for_warmup = returns_df.loc[:dates_2025.min()].tail(252) if len(dates_2025) > 0 else returns_df.tail(252)
print(f"\nDatos historicos previos disponibles para warmup: {len(hist_for_warmup)} dias")
print(f"  desde: {hist_for_warmup.index.min().date()} hasta: {hist_for_warmup.index.max().date()}")

if len(dates_2025) < 100:
    print("\nAVISO: pocos dias de trading en 2025. Esto puede afectar la fiabilidad de las metricas.")

### Reconstrucción de los `StandardScaler` usados en entrenamiento

El modelo espera inputs y outputs **normalizados** con un `StandardScaler` ajustado sobre el `X_train` exacto que se usó en `06_mixtos.ipynb`. Para garantizar que la inferencia OOS sea bit-a-bit equivalente, reconstruimos esos scalers replicando el `split_triple` original.

In [ ]:
from sklearn.model_selection import train_test_split as _tts


def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    arr = data.values if isinstance(data, pd.DataFrame) else data
    for i in range(len(arr) - input_window_size - output_window_size + 1):
        X.append(arr[i : i + input_window_size])
        y.append(np.mean(arr[i + input_window_size : i + input_window_size + output_window_size], axis=0))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def split_triple(X, y, test_size=0.1, val_size=0.1, random_state=RANDOM_SEED):
    X_train_full, X_test, y_train_full, y_test = _tts(X, y, test_size=test_size, shuffle=False)
    X_train, X_val, y_train, y_val = _tts(X_train_full, y_train_full, test_size=val_size, shuffle=True, random_state=random_state)
    return X_train, X_val, X_test, y_train, y_val, y_test


X_full, y_full = create_time_series_data(returns_df, INPUT_WINDOW, OUTPUT_WINDOW)
X_train, X_val, X_test, y_train, y_val, y_test = split_triple(X_full, y_full)

scaler_X_inference = StandardScaler()
scaler_y_inference = StandardScaler()
scaler_X_inference.fit(X_train.reshape(-1, X_train.shape[-1]))
scaler_y_inference.fit(y_train)
print(f"scaler_X mean (primeros 3): {scaler_X_inference.mean_[:3]}")
print(f"scaler_X scale (primeros 3): {scaler_X_inference.scale_[:3]}")
print(f"scaler_y mean (primeros 3): {scaler_y_inference.mean_[:3]}")
print(f"scaler_y scale (primeros 3): {scaler_y_inference.scale_[:3]}")
print(f"\nDimensiones X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
def predict_returns_top1(returns_window):
    """Devuelve predict de retornos medios a 90 dias usando model_top1.

    `returns_window`: array (90, 23) o (1, 90, 23) en log returns SIN normalizar.
    Devuelve array (23,) con la prediccion en log returns (escala real).
    """
    if model_top1 is None:
        raise RuntimeError("model_top1 no esta cargado.")
    if returns_window.ndim == 2:
        returns_window = returns_window[np.newaxis, ...]
    X_n = scaler_X_inference.transform(returns_window.reshape(-1, returns_window.shape[-1])).reshape(returns_window.shape).astype(np.float32)
    pred_n = model_top1.predict(X_n, verbose=0)
    pred_real = scaler_y_inference.inverse_transform(pred_n)
    return pred_real[0]


def predict_returns_ensemble(returns_window):
    """Devuelve la media simple de predicciones de los 3 modelos ensemble. Cada uno usa su propio scaler.

    Por simplicidad usamos el mismo scaler global (los 3 fueron entrenados con el mismo X_train).
    """
    if not ensemble_models:
        raise RuntimeError("ensemble_models vacio.")
    if returns_window.ndim == 2:
        returns_window = returns_window[np.newaxis, ...]
    X_n = scaler_X_inference.transform(returns_window.reshape(-1, returns_window.shape[-1])).reshape(returns_window.shape).astype(np.float32)
    preds = []
    for name, m in ensemble_models.items():
        pred_n = m.predict(X_n, verbose=0)
        pred_real = scaler_y_inference.inverse_transform(pred_n)[0]
        preds.append(pred_real)
    return np.mean(preds, axis=0)


if model_top1 is not None and len(dates_2025) > 0:
    first_2025 = dates_2025.min()
    window_idx_end = returns_df.index.get_loc(first_2025)
    window = returns_df.iloc[window_idx_end - INPUT_WINDOW : window_idx_end].values
    pred_top1 = predict_returns_top1(window)
    print(f"Prediccion top1 para {first_2025.date()} (90d ahead):")
    print(f"  shape: {pred_top1.shape}")
    print(f"  rango: [{pred_top1.min():.6f}, {pred_top1.max():.6f}]")
    print(f"  mean : {pred_top1.mean():.6f}")
    if ensemble_models:
        pred_ens = predict_returns_ensemble(window)
        print(f"\nPrediccion ensemble para misma fecha:")
        print(f"  rango: [{pred_ens.min():.6f}, {pred_ens.max():.6f}]")
        print(f"  mean : {pred_ens.mean():.6f}")

## 3. Funciones de pesos

Una función por estrategia, todas con la misma firma: `(mu_hist, cov_hist, pred_returns) -> weights`. Eso permite usar el mismo `backtest_strategy` para todas.

Hiperparámetros:
- `MAX_WEIGHT = 0.15` (cap por activo, evita concentración extrema).
- `MIN_WEIGHT = 0.0` (no short, todos los pesos no negativos).
- `LOOKBACK_HIST = 252` días para `μ_hist` y `Σ_hist`.
- `TOP_K = 5` para la cartera top-K.
- `DELTA_BL = 2.5` (aversión al riesgo para CAPM equilibrium en Black-Litterman).
- `TAU_BL = 0.05` (escala de incertidumbre del prior en BL).

In [ ]:
MAX_WEIGHT = 0.15
MIN_WEIGHT = 0.0
LOOKBACK_HIST = 252
TOP_K = 5
DELTA_BL = 2.5
TAU_BL = 0.05
TRANSACTION_COST_BPS = 5
RISK_FREE = 0.0


def weights_equal(mu, cov, pred, n_assets=N_ASSETS):
    """EW: pesos uniformes 1/n_assets."""
    return np.ones(n_assets) / n_assets


def _solve_min_var(cov, max_w=MAX_WEIGHT, min_w=MIN_WEIGHT):
    n = cov.shape[0]
    w0 = np.ones(n) / n
    def obj(w):
        return float(w @ cov @ w)
    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(min_w, max_w)] * n
    res = minimize(obj, w0, method="SLSQP", bounds=bounds, constraints=cons,
                   options={"maxiter": 200, "ftol": 1e-10})
    if not res.success:
        return w0
    return res.x


def weights_min_var(mu, cov, pred, n_assets=N_ASSETS):
    """Minimiza varianza sujeto a sum(w)=1, 0<=w<=MAX_WEIGHT."""
    return _solve_min_var(cov)


def _solve_max_sharpe(mu, cov, rf=RISK_FREE, max_w=MAX_WEIGHT, min_w=MIN_WEIGHT):
    n = len(mu)
    w0 = np.ones(n) / n
    def neg_sharpe(w):
        port_ret = w @ mu
        port_vol = np.sqrt(w @ cov @ w)
        if port_vol < 1e-10:
            return 0.0
        return -(port_ret - rf) / port_vol
    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(min_w, max_w)] * n
    res = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=cons,
                   options={"maxiter": 200, "ftol": 1e-10})
    if not res.success or np.isnan(res.x).any():
        return w0
    return res.x


def weights_max_sharpe_hist(mu, cov, pred, n_assets=N_ASSETS):
    """Max-Sharpe usando mu/cov historicos."""
    return _solve_max_sharpe(mu, cov)


def weights_top_k(mu, cov, pred, n_assets=N_ASSETS, k=TOP_K):
    """Top-K activos por pred return, pesos iguales."""
    order = np.argsort(pred)[-k:]
    w = np.zeros(n_assets)
    w[order] = 1.0 / k
    return w


def weights_momentum_tilt(mu, cov, pred, n_assets=N_ASSETS):
    """μ_tilted = μ_hist + λ * normalized pred. Max-Sharpe con tilted mu.

    λ = std(mu_hist) — escala las predicciones al rango natural de los retornos historicos.
    """
    pred_norm = (pred - pred.mean()) / (pred.std() + 1e-9)
    lam = mu.std()
    mu_tilted = mu + lam * pred_norm
    return _solve_max_sharpe(mu_tilted, cov)


def weights_black_litterman(mu, cov, pred, n_assets=N_ASSETS, delta=DELTA_BL, tau=TAU_BL):
    """Black-Litterman con views Q = pred.

    Prior: pi = delta * Sigma * w_market (asumimos w_market = equal-weight para simplificar).
    Views: P = identidad (una view por activo), Q = pred returns.
    Omega: diagonal con tau * diag(P @ Sigma @ P.T).
    """
    w_mkt = np.ones(n_assets) / n_assets
    pi = delta * cov @ w_mkt
    P = np.eye(n_assets)
    omega = np.diag(np.diag(tau * (P @ cov @ P.T)) + 1e-8)
    tau_cov = tau * cov
    try:
        tau_cov_inv = np.linalg.inv(tau_cov)
    except np.linalg.LinAlgError:
        return _solve_max_sharpe(mu, cov)
    try:
        omega_inv = np.linalg.inv(omega)
    except np.linalg.LinAlgError:
        return _solve_max_sharpe(mu, cov)
    posterior_cov_inv = tau_cov_inv + P.T @ omega_inv @ P
    try:
        posterior_cov = np.linalg.inv(posterior_cov_inv)
    except np.linalg.LinAlgError:
        return _solve_max_sharpe(mu, cov)
    mu_bl = posterior_cov @ (tau_cov_inv @ pi + P.T @ omega_inv @ pred)
    return _solve_max_sharpe(mu_bl, cov + posterior_cov)


STRATEGIES = {
    "EW": {"fn": weights_equal, "uses_pred": False, "rebalances": False},
    "MinVar": {"fn": weights_min_var, "uses_pred": False, "rebalances": True},
    "MaxSharpe_hist": {"fn": weights_max_sharpe_hist, "uses_pred": False, "rebalances": True},
    "TopK_pred": {"fn": weights_top_k, "uses_pred": True, "rebalances": True},
    "Momentum_tilt_pred": {"fn": weights_momentum_tilt, "uses_pred": True, "rebalances": True},
    "BlackLitterman_pred": {"fn": weights_black_litterman, "uses_pred": True, "rebalances": True},
}

print(f"Estrategias definidas: {list(STRATEGIES.keys())}")

## 4. Backtest

- `get_rebalance_dates(year, freq)` — primer día hábil de cada mes (freq='M') o trimestre ('Q').
- `backtest_strategy(weights_fn, freq, predict_fn)` — loop con realized returns + cost de transacción.
- `compute_portfolio_metrics(daily_returns)` — todas las métricas estándar.

In [ ]:
def get_rebalance_dates(year, freq, dates_index):
    """Devuelve fechas de rebalance dentro de `year` segun frecuencia M (mensual) o Q (trimestral).

    Para cada periodo, toma el primer dia HABIL disponible en `dates_index`.
    """
    year_dates = dates_index[dates_index.year == year]
    if len(year_dates) == 0:
        return []

    rebalance_dates = []
    if freq == "M":
        for month in range(1, 13):
            mdates = year_dates[year_dates.month == month]
            if len(mdates) > 0:
                rebalance_dates.append(mdates[0])
    elif freq == "Q":
        for q_month in [1, 4, 7, 10]:
            mdates = year_dates[year_dates.month == q_month]
            if len(mdates) > 0:
                rebalance_dates.append(mdates[0])
    elif freq == "Y":
        rebalance_dates.append(year_dates[0])
    return rebalance_dates


def _hist_mu_cov(returns_simple, end_date, lookback=LOOKBACK_HIST):
    """Calcula mu (media diaria) y cov (matriz de covarianza diaria) de los `lookback` dias previos a end_date."""
    idx_end = returns_simple.index.get_loc(end_date)
    start = max(0, idx_end - lookback)
    hist = returns_simple.iloc[start:idx_end]
    return hist.mean().values, hist.cov().values


def _predict_for_date(returns_log, rebalance_date, predict_fn):
    """Construye la ventana de 90 dias previa a rebalance_date y llama al predict_fn."""
    idx_end = returns_log.index.get_loc(rebalance_date)
    if idx_end < INPUT_WINDOW:
        return None
    window = returns_log.iloc[idx_end - INPUT_WINDOW : idx_end].values
    pred_log = predict_fn(window)
    return pred_log


def backtest_strategy(
    *,
    strategy_name,
    freq,
    returns_log,
    returns_simple,
    predict_fn=None,
    year=YEAR_BACKTEST,
    transaction_cost_bps=TRANSACTION_COST_BPS,
):
    """Backtest una estrategia durante `year` con la frecuencia y predict_fn dados.

    Devuelve dict con: weights_history, daily_returns, nav_series, dates_rebalance.
    """
    cfg = STRATEGIES[strategy_name]
    if not cfg["rebalances"]:
        actual_freq = "Y"
    else:
        actual_freq = freq
    rebalance_dates = get_rebalance_dates(year, actual_freq, returns_simple.index)
    if not rebalance_dates:
        return None

    year_dates = returns_simple.index[returns_simple.index.year == year]
    daily_port_rets = pd.Series(index=year_dates, dtype=float)
    weights_history = pd.DataFrame(index=rebalance_dates, columns=returns_simple.columns, dtype=float)
    predictions_history = pd.DataFrame(index=rebalance_dates, columns=returns_simple.columns, dtype=float)
    turnovers = []

    current_weights = None
    last_weights = None

    for i, reb_date in enumerate(rebalance_dates):
        mu_h, cov_h = _hist_mu_cov(returns_simple, reb_date, lookback=LOOKBACK_HIST)

        pred = None
        if cfg["uses_pred"]:
            if predict_fn is None:
                raise ValueError(f"strategy {strategy_name} requires predict_fn")
            pred = _predict_for_date(returns_log, reb_date, predict_fn)
            if pred is None:
                continue
            predictions_history.loc[reb_date] = pred

        try:
            w = cfg["fn"](mu_h, cov_h, pred)
        except Exception as exc:
            print(f"    rebalance {reb_date.date()}: error en pesos ({type(exc).__name__}: {exc}), uso EW")
            w = np.ones(N_ASSETS) / N_ASSETS

        w = np.clip(w, MIN_WEIGHT, MAX_WEIGHT)
        if w.sum() > 0:
            w = w / w.sum()

        weights_history.loc[reb_date] = w

        if last_weights is None:
            turnover = w.sum()
        else:
            turnover = np.abs(w - last_weights).sum() / 2
        turnovers.append(turnover)

        next_reb = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else None
        period_dates = year_dates[(year_dates >= reb_date) & ((year_dates < next_reb) if next_reb is not None else (year_dates >= reb_date))]

        for d in period_dates:
            day_rets = returns_simple.loc[d].values
            daily_port_rets.loc[d] = float(np.sum(w * day_rets))

        if turnover > 0:
            daily_port_rets.loc[reb_date] -= turnover * (transaction_cost_bps / 10000)

        last_weights = w
        current_weights = w

    daily_port_rets = daily_port_rets.dropna()
    nav = (1 + daily_port_rets).cumprod()
    weights_history = weights_history.dropna(how="all")

    return {
        "strategy": strategy_name,
        "freq": actual_freq,
        "daily_returns": daily_port_rets,
        "nav": nav,
        "weights_history": weights_history,
        "predictions_history": predictions_history.dropna(how="all"),
        "rebalance_dates": rebalance_dates,
        "turnovers": turnovers,
    }


def compute_portfolio_metrics(daily_returns, benchmark_returns=None):
    """Computa metricas estandar de un backtest."""
    daily_returns = pd.Series(daily_returns).dropna()
    if len(daily_returns) == 0:
        return {}
    cum = (1 + daily_returns).cumprod()
    cum_ret = float(cum.iloc[-1] - 1)
    n = len(daily_returns)
    ann_factor = 252
    ann_ret = float((1 + daily_returns.mean()) ** ann_factor - 1)
    ann_vol = float(daily_returns.std() * np.sqrt(ann_factor))
    sharpe = float((daily_returns.mean() * ann_factor - RISK_FREE) / (ann_vol + 1e-12))
    running_max = cum.cummax()
    drawdown = (cum / running_max) - 1.0
    max_dd = float(drawdown.min())

    metrics = {
        "cum_return": cum_ret,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "n_days": int(n),
    }
    if benchmark_returns is not None:
        bench = pd.Series(benchmark_returns).reindex(daily_returns.index).dropna()
        common = daily_returns.index.intersection(bench.index)
        if len(common) > 5:
            d_aligned = daily_returns.loc[common]
            b_aligned = bench.loc[common]
            active = d_aligned - b_aligned
            te = float(active.std() * np.sqrt(ann_factor))
            ir = float((active.mean() * ann_factor) / (te + 1e-12))
            metrics["tracking_error"] = te
            metrics["information_ratio"] = ir
    return metrics

print("Backtest functions definidas.")

## 5. Loop principal

Ejecutamos:
- **EW**: una vez (no se rebalancea).
- **MinVar, MaxSharpe_hist**: una vez por frecuencia (no usan predicción). 4 backtests.
- **TopK_pred, Momentum_tilt_pred, BlackLitterman_pred**: por frecuencia × modelo predicción. 12 backtests.

Total: **17 backtests**. Output guardado en `carteras_2025_resultados.csv`.

In [ ]:
FREQS = ["M", "Q"]
PRED_MODELS = {}
if model_top1 is not None:
    PRED_MODELS["top1"] = predict_returns_top1
if ensemble_models:
    PRED_MODELS["ensemble"] = predict_returns_ensemble

print(f"Frequencies: {FREQS}")
print(f"Pred models : {list(PRED_MODELS.keys())}")
print()

bench_returns = simple_returns_df[simple_returns_df.index.year == YEAR_BACKTEST].mean(axis=1)

backtest_runs = []
results_rows = []

t0_global = time.time()

for strat_name, cfg in STRATEGIES.items():
    if not cfg["uses_pred"]:
        if cfg["rebalances"]:
            for f in FREQS:
                print(f"Backtest: {strat_name:25s} freq={f}")
                bt = backtest_strategy(
                    strategy_name=strat_name, freq=f,
                    returns_log=returns_df, returns_simple=simple_returns_df,
                    predict_fn=None,
                )
                if bt is None:
                    continue
                metrics = compute_portfolio_metrics(bt["daily_returns"], benchmark_returns=bench_returns)
                row = {"strategy": strat_name, "freq": f, "pred_model": "none", **metrics}
                row["mean_turnover"] = float(np.mean(bt["turnovers"])) if bt["turnovers"] else 0.0
                row["n_rebalances"] = len(bt["rebalance_dates"])
                results_rows.append(row)
                backtest_runs.append({"key": f"{strat_name}_freq{f}_predNone", **bt, **metrics})
                print(f"  sharpe={metrics.get('sharpe', 0):+.3f}  ann_ret={metrics.get('ann_return', 0):+.2%}  ann_vol={metrics.get('ann_vol', 0):.2%}  MDD={metrics.get('max_drawdown', 0):+.2%}")
        else:
            print(f"Backtest: {strat_name:25s} freq=Y (no rebalance)")
            bt = backtest_strategy(
                strategy_name=strat_name, freq="Y",
                returns_log=returns_df, returns_simple=simple_returns_df,
                predict_fn=None,
            )
            if bt is not None:
                metrics = compute_portfolio_metrics(bt["daily_returns"], benchmark_returns=bench_returns)
                row = {"strategy": strat_name, "freq": "Y", "pred_model": "none", **metrics}
                row["mean_turnover"] = float(np.mean(bt["turnovers"])) if bt["turnovers"] else 0.0
                row["n_rebalances"] = len(bt["rebalance_dates"])
                results_rows.append(row)
                backtest_runs.append({"key": f"{strat_name}_freqY_predNone", **bt, **metrics})
                print(f"  sharpe={metrics.get('sharpe', 0):+.3f}  ann_ret={metrics.get('ann_return', 0):+.2%}  ann_vol={metrics.get('ann_vol', 0):.2%}  MDD={metrics.get('max_drawdown', 0):+.2%}")
    else:
        for f in FREQS:
            for pred_name, pred_fn in PRED_MODELS.items():
                print(f"Backtest: {strat_name:25s} freq={f}  pred={pred_name}")
                bt = backtest_strategy(
                    strategy_name=strat_name, freq=f,
                    returns_log=returns_df, returns_simple=simple_returns_df,
                    predict_fn=pred_fn,
                )
                if bt is None:
                    continue
                metrics = compute_portfolio_metrics(bt["daily_returns"], benchmark_returns=bench_returns)
                row = {"strategy": strat_name, "freq": f, "pred_model": pred_name, **metrics}
                row["mean_turnover"] = float(np.mean(bt["turnovers"])) if bt["turnovers"] else 0.0
                row["n_rebalances"] = len(bt["rebalance_dates"])
                results_rows.append(row)
                backtest_runs.append({"key": f"{strat_name}_freq{f}_pred{pred_name}", **bt, **metrics})
                print(f"  sharpe={metrics.get('sharpe', 0):+.3f}  ann_ret={metrics.get('ann_return', 0):+.2%}  ann_vol={metrics.get('ann_vol', 0):.2%}  MDD={metrics.get('max_drawdown', 0):+.2%}")

print()
print(f"Total backtests ejecutados: {len(backtest_runs)} en {(time.time()-t0_global)/60:.1f} min")

In [ ]:
df_results = pd.DataFrame(results_rows)
if len(df_results) > 0:
    cols_order = ["strategy", "freq", "pred_model", "cum_return", "ann_return", "ann_vol", "sharpe",
                  "max_drawdown", "tracking_error", "information_ratio", "mean_turnover", "n_rebalances", "n_days"]
    cols_existing = [c for c in cols_order if c in df_results.columns]
    df_results = df_results[cols_existing]
    df_results.to_csv(RESULTS_DIR / "carteras_2025_resultados.csv", index=False)
    print(f"Guardado: {RESULTS_DIR / 'carteras_2025_resultados.csv'}")
    print()
    print(df_results.to_string(index=False))

returns_df_export = pd.DataFrame({bt["key"]: bt["daily_returns"] for bt in backtest_runs}).reset_index()
returns_df_export.columns = ["date"] + list(returns_df_export.columns[1:])
returns_df_export.to_csv(RESULTS_DIR / "carteras_2025_returns.csv", index=False)
print(f"\nGuardado: {RESULTS_DIR / 'carteras_2025_returns.csv'}")

weights_rows = []
for bt in backtest_runs:
    if bt["weights_history"] is None or len(bt["weights_history"]) == 0:
        continue
    wh = bt["weights_history"].copy()
    wh["strategy_key"] = bt["key"]
    wh = wh.reset_index().rename(columns={"index": "date"})
    weights_rows.append(wh)
if weights_rows:
    weights_export = pd.concat(weights_rows, ignore_index=True)
    weights_export.to_csv(RESULTS_DIR / "carteras_2025_pesos.csv", index=False)
    print(f"Guardado: {RESULTS_DIR / 'carteras_2025_pesos.csv'}")

## 6. Visualización

Tres gráficas clave:
1. **Evolución del NAV** durante 2025 (todas las carteras, una línea por cada).
2. **Bar chart de métricas** (Sharpe y MDD comparados).
3. **Heatmap de pesos por activo** a lo largo del año para las carteras con predicción.

In [ ]:
if backtest_runs:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    palette = plt.get_cmap("tab20")

    ax = axes[0]
    for i, bt in enumerate(backtest_runs):
        if "M" not in bt["freq"] and bt["freq"] != "Y":
            continue
        ax.plot(bt["nav"].index, bt["nav"].values, label=bt["key"], color=palette(i % 20), linewidth=1.2, alpha=0.85)
    ax.axhline(1, color="black", linestyle=":", linewidth=0.8)
    ax.set_title(f"Evolucion NAV carteras 2025 - rebalance mensual + EW")
    ax.set_ylabel("NAV (1 = inicio anio)")
    ax.legend(fontsize=6, loc="best", ncol=1)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    for i, bt in enumerate(backtest_runs):
        if "Q" not in bt["freq"] and bt["freq"] != "Y":
            continue
        ax.plot(bt["nav"].index, bt["nav"].values, label=bt["key"], color=palette(i % 20), linewidth=1.2, alpha=0.85)
    ax.axhline(1, color="black", linestyle=":", linewidth=0.8)
    ax.set_title(f"Evolucion NAV carteras 2025 - rebalance trimestral + EW")
    ax.set_ylabel("NAV (1 = inicio anio)")
    ax.legend(fontsize=6, loc="best", ncol=1)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "carteras_2025_evolucion.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"guardado: {RESULTS_DIR / 'carteras_2025_evolucion.png'}")

In [ ]:
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(18, 6))
    df_plot = df_results.copy()
    df_plot["label"] = df_plot.apply(lambda r: f"{r['strategy']}\n[{r['freq']}|{r['pred_model']}]", axis=1)
    df_plot = df_plot.sort_values("sharpe", ascending=True)

    metrics_to_plot = [
        ("sharpe", "Sharpe ratio", "tab:blue"),
        ("ann_return", "Ann. return", "tab:green"),
        ("ann_vol", "Ann. volatility", "tab:orange"),
        ("max_drawdown", "Max drawdown", "tab:red"),
    ]

    for ax, (col, title, color) in zip(axes, metrics_to_plot):
        ax.barh(range(len(df_plot)), df_plot[col].values, color=color)
        ax.set_yticks(range(len(df_plot)))
        ax.set_yticklabels(df_plot["label"], fontsize=7)
        ax.set_title(title)
        ax.axvline(0, color="black", linewidth=0.8)
        for i, v in enumerate(df_plot[col].values):
            if isinstance(v, (int, float)) and not np.isnan(v):
                fmt = "{:+.2%}" if "return" in col or "drawdown" in col or "vol" in col else "{:+.3f}"
                ax.text(v, i, " " + fmt.format(v), va="center", fontsize=6)

    fig.suptitle("Comparativa de metricas - carteras 2025", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(RESULTS_DIR / "carteras_2025_metricas.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"guardado: {RESULTS_DIR / 'carteras_2025_metricas.png'}")

In [ ]:
pred_strategies = [s for s, c in STRATEGIES.items() if c["uses_pred"]]
pred_runs = [bt for bt in backtest_runs if bt["strategy"] in pred_strategies]

if pred_runs:
    n_plots = len(pred_runs)
    ncols = min(3, n_plots)
    nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for ax_i, bt in enumerate(pred_runs):
        ax = axes_flat[ax_i]
        wh = bt["weights_history"]
        if wh is None or len(wh) == 0:
            ax.axis("off")
            continue
        data = wh.fillna(0).values.T
        im = ax.imshow(data, aspect="auto", cmap="YlOrRd", vmin=0, vmax=MAX_WEIGHT)
        ax.set_xticks(range(len(wh)))
        ax.set_xticklabels([d.strftime("%Y-%m") for d in wh.index], rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(wh.columns)))
        ax.set_yticklabels(wh.columns, fontsize=7)
        ax.set_title(bt["key"], fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="peso")

    for ax_i in range(len(pred_runs), len(axes_flat)):
        axes_flat[ax_i].axis("off")

    fig.suptitle("Heatmap de pesos por activo a lo largo de 2025 (carteras con predicciones)", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(RESULTS_DIR / "carteras_2025_pesos_heatmap.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"guardado: {RESULTS_DIR / 'carteras_2025_pesos_heatmap.png'}")

## 7. Conclusiones

Generadas automáticamente desde los resultados. Aprovechables directamente para la presentación oral (sección "Carteras 2025").

In [ ]:
conclusiones = []

if len(df_results) > 0:
    n_runs = len(df_results)
    conclusiones.append(f"Se ejecutaron {n_runs} backtests durante {YEAR_BACKTEST}.")

    by_sharpe = df_results.sort_values("sharpe", ascending=False)
    top1 = by_sharpe.iloc[0]
    conclusiones.append(
        f"Mejor Sharpe: {top1['strategy']} (freq={top1['freq']}, pred={top1['pred_model']}) -> "
        f"Sharpe={top1['sharpe']:+.3f}, ann_ret={top1['ann_return']:+.2%}, MDD={top1['max_drawdown']:+.2%}."
    )

    sin_pred = df_results[df_results["pred_model"] == "none"]
    con_pred = df_results[df_results["pred_model"] != "none"]
    if len(sin_pred) > 0 and len(con_pred) > 0:
        sharpe_sin = sin_pred["sharpe"].max()
        sharpe_con = con_pred["sharpe"].max()
        delta = sharpe_con - sharpe_sin
        if delta > 0.05:
            conclusiones.append(f"Las carteras con predicciones del modelo mejoran Sharpe en +{delta:.3f} respecto al mejor baseline sin predicciones.")
        elif delta < -0.05:
            conclusiones.append(f"Las predicciones del modelo NO superan al mejor baseline sin predicciones (delta Sharpe = {delta:+.3f}). El modelo no genero alpha en 2025.")
        else:
            conclusiones.append(f"Las predicciones tienen impacto neutro en Sharpe (delta = {delta:+.3f}). El modelo replica casi exactamente el comportamiento de los baselines.")

    mensual = df_results[df_results["freq"] == "M"]
    trimestral = df_results[df_results["freq"] == "Q"]
    if len(mensual) > 0 and len(trimestral) > 0:
        sharpe_m = mensual["sharpe"].mean()
        sharpe_q = trimestral["sharpe"].mean()
        if sharpe_m > sharpe_q + 0.05:
            conclusiones.append(f"Rebalance mensual gana en Sharpe medio ({sharpe_m:+.3f}) sobre trimestral ({sharpe_q:+.3f}).")
        elif sharpe_q > sharpe_m + 0.05:
            conclusiones.append(f"Rebalance trimestral gana en Sharpe medio ({sharpe_q:+.3f}) sobre mensual ({sharpe_m:+.3f}). Menos rebalanceo -> menos coste de transaccion.")
        else:
            conclusiones.append(f"Mensual vs trimestral: diferencia de Sharpe medio <0.05, dominante por turnover.")

    if "top1" in df_results["pred_model"].values and "ensemble" in df_results["pred_model"].values:
        top1_sharpe = df_results[df_results["pred_model"] == "top1"]["sharpe"].max()
        ens_sharpe = df_results[df_results["pred_model"] == "ensemble"]["sharpe"].max()
        if abs(top1_sharpe - ens_sharpe) < 0.05:
            conclusiones.append(f"Top-1 ({top1_sharpe:+.3f}) y ensemble ({ens_sharpe:+.3f}) producen Sharpes equivalentes en cartera. El extra del ensemble no compensa en backtesting.")
        elif ens_sharpe > top1_sharpe:
            conclusiones.append(f"Ensemble Sharpe ({ens_sharpe:+.3f}) supera a top-1 ({top1_sharpe:+.3f}) en cartera.")
        else:
            conclusiones.append(f"Top-1 Sharpe ({top1_sharpe:+.3f}) supera al ensemble ({ens_sharpe:+.3f}). El modelo top-1 es suficiente para production.")

    ranking = by_sharpe[["strategy", "freq", "pred_model", "sharpe", "ann_return", "max_drawdown"]].reset_index(drop=True)
    print("Ranking carteras 2025 por Sharpe:")
    print(ranking.to_string(index=False))

print("\nConclusiones automaticas:")
print("=" * 80)
for i, c in enumerate(conclusiones, 1):
    print(f"{i:2d}. {c}")

### Reflexión cualitativa para la presentación

- **EW como referencia**: en mercados eficientes con activos correlacionados (23 grandes blue chips del SP500), la cartera equiponderada es **muy difícil de batir** sin información extra. Es el benchmark honesto.
- **MinVar y MaxSharpe histórico**: dependen de cuán estable sea el régimen de volatilidad/correlación 252 días antes vs durante 2025. Si hay un shift de régimen, fallan.
- **Top-K con predicciones**: depende crucialmente de la **calidad direccional** de las predicciones (no de la magnitud absoluta). Si el modelo acierta el orden relativo de los 23 activos, Top-K funciona.
- **Momentum tilt**: combina la `μ_hist` (estable) con la predicción (informativa). Más robusto que Top-K cuando las predicciones son ruidosas.
- **Black-Litterman**: el más "elegante" matemáticamente pero el más sensible a la calibración de `τ` y `Ω`. Tiende a quedarse cerca del prior (equilibrium).

### Para la oral

1. Mostrar la **tabla de Sharpe ordenada** y destacar la mejor cartera.
2. Mostrar la **gráfica de evolución NAV** comparando EW vs la mejor con predicción.
3. Comentar si las predicciones generaron alpha real o no.
4. Reflexión final: en horizonte de 1 año con 23 blue chips, las diferencias suelen ser **pequeñas pero defendibles**. El profesor valora el rigor del proceso más que el resultado numérico.

## 8. Validación final + cheat sheet

In [ ]:
checks_total = 0
checks_ok = 0


def _do(label, cond, detail=""):
    global checks_total, checks_ok
    checks_total += 1
    icono = "[OK]" if cond else "[FAIL]"
    print(f"{icono} {label} {detail}")
    if cond:
        checks_ok += 1


print("Validacion del notebook 08")
print("=" * 80)

_do("CSV carteras_2025_resultados.csv existe", (RESULTS_DIR / "carteras_2025_resultados.csv").exists())
if (RESULTS_DIR / "carteras_2025_resultados.csv").exists():
    df_check = pd.read_csv(RESULTS_DIR / "carteras_2025_resultados.csv")
    _do("CSV >= 12 filas (cobertura razonable)", len(df_check) >= 12, f"(actual: {len(df_check)})")
    cols_obligatorias = {"strategy", "freq", "pred_model", "sharpe", "ann_return", "ann_vol", "max_drawdown"}
    _do("CSV tiene columnas requeridas", cols_obligatorias.issubset(set(df_check.columns)),
        f"(faltan: {cols_obligatorias - set(df_check.columns)})")

_do("CSV returns existe", (RESULTS_DIR / "carteras_2025_returns.csv").exists())
_do("CSV pesos existe", (RESULTS_DIR / "carteras_2025_pesos.csv").exists())

for fname in ["carteras_2025_evolucion.png", "carteras_2025_metricas.png", "carteras_2025_pesos_heatmap.png"]:
    _do(f"Figura {fname}", (RESULTS_DIR / fname).exists())

_do("Modelo top-1 cargable", model_top1 is not None)
_do("Ensemble (3 modelos) cargable", len(ensemble_models) == 3, f"(actual: {len(ensemble_models)})")

if len(df_results) > 0:
    n_strats_distintas = df_results["strategy"].nunique()
    _do("6 estrategias presentes", n_strats_distintas == 6, f"(actual: {n_strats_distintas})")

print("=" * 80)
print(f"Resultado: {checks_ok}/{checks_total} checks superados.")

### Cheat sheet

**Cambiar el modelo de predicción** (por ejemplo, usar uno de los modelos de investigación):

```python
mp = MODELS_DIR / "inv_B3_ewma_vol_channel_profMIX_in90_out90.keras"
custom_model = load_model(mp, compile=False)

def predict_returns_inv(window):
    if window.ndim == 2:
        window = window[np.newaxis, ...]
    X_n = scaler_X_inference.transform(window.reshape(-1, window.shape[-1])).reshape(window.shape).astype(np.float32)
    pred = custom_model.predict(X_n, verbose=0)
    return scaler_y_inference.inverse_transform(pred)[0]

PRED_MODELS["inv_ewma"] = predict_returns_inv
```

**Cambiar el rebalance** (semanal por ejemplo):

```python
def get_rebalance_dates_weekly(year, dates_index):
    year_dates = dates_index[dates_index.year == year]
    return list(year_dates[year_dates.weekday == 0])
```

**Cambiar k en top-K**:

```python
TOP_K = 10
STRATEGIES["TopK10_pred"] = {"fn": lambda mu, cov, pred: weights_top_k(mu, cov, pred, k=10), "uses_pred": True, "rebalances": True}
```

**Re-ejecutar todo desde cero**: borrar `results/carteras_2025_*.csv` y ejecutar todas las celdas.